In [1]:
!pip install --upgrade binance-historical-data plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.1 MB/s eta 0:00:00


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [3]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data

In [4]:
ticker = 'BTCUSDT'
data_frequency = '15m'
download_ticker(ticker, "2025-06-01", "2025-07-17", data_frequency)
data = process_csv_files(ticker, data_frequency)
data = data.drop(columns=['close_time',	'quote_asset_volume',	'number_of_trades',	'taker_buy_base_asset_volume',	'taker_buy_quote_asset_volume',	'ignore'])
data
data['hlc3'] = (data['High'] + data['Low'] + data['Close']) / 3
# Tính giá trị giao dịch (Typical Price x Volume)
# Chuẩn bị sẵn tử số cho công thức VWAP
data['tp_vol'] = data['hlc3'] * data['volume']


---> Found overall tickers: 589
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 15m
---> Start Date: 20250601
---> End Date: 20250717


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/1 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/17 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download:   0%|          | 0/17 [00:00<?, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 1 months 17 days
---> For BTCUSD new data saved for: 0 months 0 days


In [5]:
def backtest_vwap(df, a, b, fee_rate=0.00075):
    """
    Thực hiện backtest cho chiến lược giao cắt Rolling VWAP (MVWAP).
    Đây là chiến lược Long/Short, có tính đến phí giao dịch.

    Args:
        df (pd.DataFrame): DataFrame dữ liệu.
        a (int): Cửa sổ của MVWAP nhanh.
        b (int): Cửa sổ của MVWAP chậm.
        fee_rate (float): Tỷ lệ phí giao dịch (ví dụ: 0.075% = 0.00075).

    Returns:
        pd.Series: Chuỗi lợi nhuận của chiến lược sau khi đã trừ phí.
    """
    # 1. Tính toán hai đường MVWAP
    mvwap_fast = df['tp_vol'].rolling(window=a).sum() / (df['volume'].rolling(window=a).sum() + 1e-9)
    mvwap_slow = df['tp_vol'].rolling(window=b).sum() / (df['volume'].rolling(window=b).sum() + 1e-9)

    # 2. Tạo tín hiệu Long/Short (-1 = Short, 1 = Long)
    signal = np.sign(mvwap_fast - mvwap_slow)

    # 3. Tính toán lợi nhuận thị trường và dịch chuyển tín hiệu
    market_returns = df['Close'].pct_change()
    shifted_signal = signal.shift(1).fillna(0)

    # 4. Tính lợi nhuận chiến lược (chưa có phí)
    strategy_returns = market_returns * shifted_signal

    # 5. TÍNH TOÁN PHÍ GIAO DỊCH
    # Một giao dịch xảy ra khi tín hiệu thay đổi
    # .diff() sẽ khác 0 khi signal thay đổi (ví dụ: từ 1 sang -1, .diff() = -2)
    trades = shifted_signal.diff().ne(0)
    trade_count = trades.sum()
    trade_times = df['open_time'][trades].tolist()
    print (trade_count)
    print (trade_times)

    # Trừ phí vào lợi nhuận tại những cây nến có giao dịch
    strategy_returns[trades] -= fee_rate

    return strategy_returns.fillna(0)

In [6]:
backtest_vwap(data, 38, 44)

179
[Timestamp('2025-06-01 00:15:00'), Timestamp('2025-06-01 11:15:00'), Timestamp('2025-06-01 12:15:00'), Timestamp('2025-06-01 13:30:00'), Timestamp('2025-06-01 16:30:00'), Timestamp('2025-06-02 07:45:00'), Timestamp('2025-06-02 21:30:00'), Timestamp('2025-06-03 09:15:00'), Timestamp('2025-06-03 14:45:00'), Timestamp('2025-06-04 00:45:00'), Timestamp('2025-06-04 10:00:00'), Timestamp('2025-06-04 11:45:00'), Timestamp('2025-06-04 23:45:00'), Timestamp('2025-06-05 01:15:00'), Timestamp('2025-06-05 06:00:00'), Timestamp('2025-06-05 08:00:00'), Timestamp('2025-06-05 09:15:00'), Timestamp('2025-06-05 10:30:00'), Timestamp('2025-06-05 13:45:00'), Timestamp('2025-06-05 14:00:00'), Timestamp('2025-06-05 15:45:00'), Timestamp('2025-06-05 17:45:00'), Timestamp('2025-06-06 06:30:00'), Timestamp('2025-06-07 01:00:00'), Timestamp('2025-06-07 05:30:00'), Timestamp('2025-06-08 04:15:00'), Timestamp('2025-06-08 13:30:00'), Timestamp('2025-06-08 15:00:00'), Timestamp('2025-06-08 15:30:00'), Timestamp

,0
1615,0.000000
1616,0.000000
1617,-0.000000
1618,0.000000
1619,-0.000000
...,...
470,-0.003347
471,-0.000296
472,-0.000963
473,-0.002107
